In [3]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

sns.set_theme(style="whitegrid")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abdou\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abdou\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abdou\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
df = pd.read_csv("../data/raw/Enron.csv")
df["subject"] = df["subject"].fillna("")
df["text"] = df["subject"] + " " + df["body"]
df["classe"] = df["label"].map({0: "Non-spam", 1: "Spam"})
print(df.shape)

(29767, 5)


In [5]:
stop_words = set(stopwords.words("english"))
stop_words.update({"nbsp", "br"})   # résidus HTML repérés à l'exploration

def nettoyer(texte):
    texte = texte.lower()                                # 1. minuscules
    texte = re.sub(r"<[^>]+>", " ", texte)               # 2. balises HTML
    texte = re.sub(r"http\S+|www\S+", " url ", texte)    # 3. adresses web
    texte = re.sub(r"\d+", " nombre ", texte)            # 4. nombres -> "nombre"
    texte = re.sub(r"[^a-z\s]", " ", texte)              # 5. caractères spéciaux et ponctuation
    texte = re.sub(r"\s+", " ", texte).strip()           # 6. espaces multiples
    tokens = texte.split()                               # 7. tokenisation
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]  # 8. stop words
    return tokens

In [6]:
exemple = df[df["label"] == 1].iloc[5]["text"]

print("=== AVANT ===")
print(exemple[:500])

print("\n=== APRÈS ===")
print(nettoyer(exemple)[:60])

print("\n=== Tokenisation NLTK sur le texte brut (pour comparaison) ===")
print(word_tokenize(exemple)[:30])

=== AVANT ===
my site links to your site now hello my name is anthony lewis . i am seeking out possible link partners that our
visitors would be interesting in visiting . i ' ve found your website to be a very good fit
for our visitors . i have already gone ahead and added your link to our website at :
i am contacting you to see if it is ok to have done so . also , i would like to ask if
you mind linking back to us ? if so , please use the linking details below and send
me the location of our link on you

=== APRÈS ===
['site', 'links', 'site', 'hello', 'name', 'anthony', 'lewis', 'seeking', 'possible', 'link', 'partners', 'visitors', 'would', 'interesting', 'visiting', 'found', 'website', 'good', 'fit', 'visitors', 'already', 'gone', 'ahead', 'added', 'link', 'website', 'contacting', 'see', 'ok', 'done', 'also', 'would', 'like', 'ask', 'mind', 'linking', 'back', 'us', 'please', 'use', 'linking', 'details', 'send', 'location', 'link', 'website', 'linking', 'details', 'title', 'acai', '

In [7]:
df["tokens"] = df["text"].apply(nettoyer)
df["text_clean"] = df["tokens"].apply(lambda t: " ".join(t))
df.head()

,subject,body,label,text,classe,tokens,text_clean
0,"hpl nom for may 25 , 2001",( see attached file : hplno 525 . xls )\r\n- h...,0,"hpl nom for may 25 , 2001 ( see attached file ...",Non-spam,"[hpl, nom, may, nombre, nombre, see, attached,...",hpl nom may nombre nombre see attached file hp...
1,re : nom / actual vols for 24 th,- - - - - - - - - - - - - - - - - - - - - - fo...,0,re : nom / actual vols for 24 th - - - - - - -...,Non-spam,"[nom, actual, vols, nombre, th, forwarded, sab...",nom actual vols nombre th forwarded sabrae zaj...
2,"enron actuals for march 30 - april 1 , 201","estimated actuals\r\nmarch 30 , 2001\r\nno flo...",0,"enron actuals for march 30 - april 1 , 201 est...",Non-spam,"[enron, actuals, march, nombre, april, nombre,...",enron actuals march nombre april nombre nombre...
3,"hpl nom for may 30 , 2001",( see attached file : hplno 530 . xls )\r\n- h...,0,"hpl nom for may 30 , 2001 ( see attached file ...",Non-spam,"[hpl, nom, may, nombre, nombre, see, attached,...",hpl nom may nombre nombre see attached file hp...
4,"hpl nom for june 1 , 2001",( see attached file : hplno 601 . xls )\r\n- h...,0,"hpl nom for june 1 , 2001 ( see attached file ...",Non-spam,"[hpl, nom, june, nombre, nombre, see, attached...",hpl nom june nombre nombre see attached file h...


In [8]:
df["nb_mots_avant"] = df["text"].str.split().str.len()
df["nb_mots_apres"] = df["tokens"].str.len()

print("Mots avant nettoyage :", df["nb_mots_avant"].sum())
print("Mots après nettoyage :", df["nb_mots_apres"].sum())
print("Réduction : {:.1f} %".format(100 * (1 - df["nb_mots_apres"].sum() / df["nb_mots_avant"].sum())))

vides = (df["nb_mots_apres"] == 0).sum()
print("\nEmails vides après nettoyage :", vides)
print(df.groupby("classe")[["nb_mots_avant", "nb_mots_apres"]].median())

Mots avant nettoyage : 9237265
Mots après nettoyage : 4567137
Réduction : 50.6 %

Emails vides après nettoyage : 1
          nb_mots_avant  nb_mots_apres
classe                                
Non-spam          175.0           83.0
Spam              140.0           69.0


In [9]:
from collections import Counter

compteur = Counter(t for tokens in df["tokens"] for t in tokens)
print("Vocabulaire (mots distincts) :", len(compteur))
print("Mots apparaissant une seule fois :", sum(1 for c in compteur.values() if c == 1))
print("\nLes 20 mots les plus fréquents :")
print(compteur.most_common(20))

Vocabulaire (mots distincts) : 143285
Mots apparaissant une seule fois : 70681

Les 20 mots les plus fréquents :
[('nombre', 461692), ('enron', 53850), ('ect', 34734), ('com', 21776), ('company', 18772), ('please', 18634), ('hou', 16887), ('would', 14191), ('subject', 13452), ('new', 12952), ('may', 11610), ('information', 11199), ('time', 10972), ('gas', 10778), ('one', 10344), ('energy', 10156), ('business', 9951), ('us', 9885), ('said', 9702), ('http', 9638)]


In [10]:
import os
os.makedirs("../data/processed", exist_ok=True)

df[["text_clean", "label"]].to_csv("../data/processed/enron_clean.csv", index=False)
print("Sauvegardé :", df.shape[0], "emails")

Sauvegardé : 29767 emails


In [11]:
vide = df[df["nb_mots_apres"] == 0]
print(vide[["subject", "body", "label"]])

                                                subject  \
4086  ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ? ...   

                                       body  label  
4086  ???? ??? ????? ?????? ?????? ????\r\n      1  


In [12]:
df = df[df["nb_mots_apres"] > 0].reset_index(drop=True)
df[["text_clean", "label"]].to_csv("../data/processed/enron_clean.csv", index=False)
print("Sauvegardé :", df.shape[0], "emails")

Sauvegardé : 29766 emails
